In [ ]:
%pip install -q otter-grader

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

# In-Class Exercise: Dimensionality Reduction with PCA and t-SNE

**DS701 — Session 16 (Wed Oct 28, 2026)**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Course-Notes-FA26/blob/main/class_activity_notebooks/11-InClass-Exercise-Dimensionality-Reduction/11-InClass-Exercise-Dimensionality-Reduction.ipynb)

**Time: 60 minutes**

**Instructions:**

- Work in groups of 2-3 people.
- Complete the code cells below by filling in the missing parts.
- Parts marked **(autograded)** are submitted to Gradescope; the open-ended parts
  are graded for participation.
- You may use AI assistance, but you must be able to **explain and justify every
  part of your solution** when asked — staff will circulate and cold-call.
- After completing the exercise, pair up with another group to review and compare
  your solutions.

| Part | Topic | Time |
|---|---|---|
| 1 | Standardize the data **(autograded)** | 5 min |
| 2 | PCA to 2 components **(autograded)** | 5 min |
| 3 | Visualize the PCA projection | 5 min |
| 4 | Explained variance and choosing $d$ **(autograded)** | 12 min |
| 5 | PCA is the SVD of the centered data **(autograded)** | 10 min |
| 6 | PCA vs. t-SNE on digits (open-ended) | 15 min |
| 7 | Perplexity sweep + peer review | 8 min |

---

## Setup: Load the Data

We'll use the **Wine** dataset for Parts 1-5: 178 samples, 13 features (chemical
properties measured in wildly different units), 3 wine classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Load the wine dataset
wine = load_wine()
X = wine.data
y = wine.target

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Classes: {wine.target_names}")

# Feature scales differ by orders of magnitude -- note this before Part 1
print("\nPer-feature variances (unscaled):")
for name, var in zip(wine.feature_names, X.var(axis=0)):
    print(f"  {name:32s} {var:12.3f}")

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/11-InClass-Exercise-Dimensionality-Reduction/tests/"
    for _q in ("q2", "q3", "q6", "q9"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

## Part 1: Data Preprocessing

Look at the per-feature variances printed above. `proline` has a variance in the
hundreds of thousands; `nonflavanoid_phenols` has a variance around 0.01.

<!-- BEGIN QUESTION -->

**Discussion (Part 1a).** PCA maximizes variance. What would happen to the first
principal component if you ran PCA on this data *without* standardizing?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Task (Part 1b, autograded).** Standardize the data so every feature has mean 0
and standard deviation 1. Store the result in `X_scaled`.

In [ ]:
scaler = StandardScaler()
X_scaled = ...

print(f"Mean of first feature (should be ~0): {X_scaled[:, 0].mean():.6f}")
print(f"Std  of first feature (should be ~1): {X_scaled[:, 0].std():.6f}")

In [ ]:
grader.check("q2")

## Part 2 (autograded): Apply PCA

**Task:** reduce the standardized data from 13 dimensions to 2 for visualization.
Fit a `PCA` object named `pca2` and store the projected data in `X_pca`.

In [ ]:
...

explained_var = pca2.explained_variance_ratio_
print(f"Reduced data shape: {X_pca.shape}")
print(f"Explained variance by PC1: {explained_var[0]:.3f}")
print(f"Explained variance by PC2: {explained_var[1]:.3f}")
print(f"Total explained variance:  {explained_var.sum():.3f}")

In [ ]:
grader.check("q3")

## Part 3 (participation): Visualize the PCA projection

<!-- BEGIN QUESTION -->

**Task:** complete the scatter plot so points are colored by wine class, and label
the axes with the fraction of variance each component explains.

In [ ]:
plt.figure(figsize=(8, 6))
...
plt.title('Wine Dataset - PCA Projection')
plt.colorbar(scatter, label='Wine Class')
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Discussion question:** are the three wine classes well separated in this 2-D
projection? Notice that PCA never saw the labels `y` — it only maximized variance.
Why did the classes separate anyway?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part 4 (autograded): Explained variance and choosing $d$

Recall from lecture: with covariance eigenvalues $\lambda_1 \ge \dots \ge \lambda_n$
and total variance $T = \sum_i \lambda_i$, the **explained variance ratio** of
component $i$ is $\lambda_i / T$, and the **cumulative explained variance**
through $k$ is $\sum_{i=1}^{k} \lambda_i / T$.

**Task:** fit PCA keeping *all* components, then compute

- `pca_full` — a `PCA` object fit to `X_scaled` with all components kept,
- `evr` — the explained variance ratio of every component (a length-13 array),
- `cum_evr` — the cumulative explained variance (a length-13 array),
- `n_components_90` — the **smallest** number of components whose cumulative
  explained variance is at least 0.90 (an `int`).

Do not hardcode the answer — compute it from `evr`.

In [ ]:
...

print(f"Explained variance ratios:\n{np.round(evr, 4)}")
print(f"\nCumulative:\n{np.round(cum_evr, 4)}")
print(f"\nComponents needed for >= 90% of the variance: {n_components_90}")
print(f"That is {n_components_90}/{X.shape[1]} of the original dimensions, "
      f"retaining {cum_evr[n_components_90 - 1]:.1%} of the variance.")

In [ ]:
grader.check("q6")

<!-- BEGIN QUESTION -->

**Task (participation):** draw the scree plot and the cumulative explained variance
curve, with a horizontal line at 0.90 and a vertical line at `n_components_90`.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
comps = np.arange(1, len(evr) + 1)

axs[0].plot(comps, evr, marker='o', linestyle='--')
axs[0].set_title('Scree Plot')
axs[0].set_ylabel('Explained\nVariance Ratio')
axs[0].grid(True)

...
axs[1].set_title('Cumulative Explained Variance')
axs[1].set_xlabel('Principal Component')
axs[1].set_ylabel('Cumulative\nExplained Variance')
axs[1].grid(True)

plt.tight_layout()
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Discussion question:** where is the "elbow" in the scree plot, and does it agree
with the 90% rule? Which criterion would you defend to a colleague, and why?
Remember from lecture that a *low*-variance direction can still be the one that
separates your classes — variance is not the same thing as usefulness.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part 5 (autograded): PCA *is* the SVD of the centered data

From lecture: if the centered data matrix has SVD $X = U\Sigma V^{T}$, then

$$X^{T}X = V\Sigma^{2}V^{T},$$

so the principal components are the columns of $V$ and the covariance eigenvalues
are $\lambda_i = \sigma_i^{2}/(m-1)$.

**Task:** verify this numerically, without calling `PCA` again.

- `U, s, Vt = np.linalg.svd(X_scaled, full_matrices=False)`
- `lambdas_svd` — the covariance eigenvalues recovered from `s` (length 13)
- `Y_svd` — the projection of `X_scaled` onto the first 2 principal directions,
  computed from `Vt` (shape `(178, 2)`)

In [ ]:
m = X_scaled.shape[0]
...

print(f"Eigenvalues from SVD:     {np.round(lambdas_svd[:4], 4)}")
print(f"Eigenvalues from sklearn: {np.round(pca_full.explained_variance_[:4], 4)}")
print(f"\nExplained variance ratio from SVD: "
      f"{np.round(lambdas_svd[:4] / lambdas_svd.sum(), 4)}")

In [ ]:
grader.check("q9")

<!-- BEGIN QUESTION -->

**Discussion question:** the assertion above compares `np.abs(Y_svd)` to
`np.abs(X_pca)` rather than the raw values. Why is the sign of a principal
component arbitrary? And why do libraries compute PCA this way instead of forming
$S = \frac{1}{m-1}X^{T}X$ and eigendecomposing it?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Part 6 (open-ended, participation): PCA vs. t-SNE on the same dataset

Now switch to the **digits** dataset — 1797 handwritten digits, each an 8x8 image
flattened to a 64-dimensional vector, with 10 classes. This is the same dataset
from the lecture.

You will embed the *same data* two ways and compare:

1. **PCA** to 2 components — linear, deterministic, preserves global variance.
2. **t-SNE** to 2 components — nonlinear, stochastic, preserves local neighborhoods.
   Follow the best practice from lecture: **PCA down to 50 dimensions first**, then
   run t-SNE on that.

> **You will be cold-called on Part 6.** Agree as a group on your reading of these
> two pictures and be ready to *justify* it — not just "t-SNE looks better," but
> which specific features of each plot you trust, which you don't, and why.

In [ ]:
digits = load_digits()
X_dig = digits.data
y_dig = digits.target
print(f"Digits shape: {X_dig.shape}, classes: {np.unique(y_dig)}")

# Standardize (t-SNE and PCA are both scale sensitive)
X_dig_scaled = StandardScaler().fit_transform(X_dig)

<!-- BEGIN QUESTION -->

**Task:** build the two embeddings. Store the 2-D PCA embedding in `X_dig_pca`
(from a `PCA` object named `pca_dig`), the 50-dimensional PCA representation in
`X_dig_pca50`, and the 2-D t-SNE embedding of `X_dig_pca50` in `X_dig_tsne`. Use
`random_state=42` throughout so your plots are reproducible.

In [ ]:
...

print(f"PCA embedding:   {X_dig_pca.shape}")
print(f"t-SNE embedding: {X_dig_tsne.shape}")
print(f"Variance kept by the 2 PCA components: "
      f"{pca_dig.explained_variance_ratio_.sum():.1%}")

<!-- END QUESTION -->



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, emb, name in [(axes[0], X_dig_pca, 'PCA'), (axes[1], X_dig_tsne, 't-SNE')]:
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=y_dig, cmap='tab10', s=12, alpha=0.8)
    ax.set_title(f'Digits - {name}')
    ax.set_xlabel(f'{name} component 1')
    ax.set_ylabel(f'{name} component 2')

fig.colorbar(sc, ax=axes, label='Digit')
plt.show()

<!-- BEGIN QUESTION -->

### Write up your group's comparison

Answer each of these in a sentence or two. Be specific — point at digits and
regions of the plots, not just "it separates better."

1. **Separation.** Which digits does PCA fail to separate that t-SNE does separate?
   Are there any digits that *both* methods confuse? What does that tell you about
   the data as opposed to the method?
2. **What you trust.** In the t-SNE plot, one cluster is visibly larger and more
   spread out than another. Is that a real property of the data? What about the
   gap between two clusters? Justify using how t-SNE's objective works.
3. **Interpretability.** PC1 for the digits is a specific direction in 64-D pixel
   space — you could reshape it to 8x8 and look at it. Can you do the same for a
   t-SNE component? Why or why not?
4. **Choosing a method.** You need a reduced representation to feed a downstream
   classifier on 50,000 new images. Which method, and why is the other one
   disqualified?
5. **Reproducibility.** We passed `random_state=42` to `TSNE`. What would change if
   we hadn't, and what would *not* change if the structure is real?

*Your group's answers:*

1. **Separation.**

2. **What you trust.**

3. **Interpretability.**

4. **Choosing a method.**

5. **Reproducibility.**

<!-- END QUESTION -->

## Part 7: Perplexity sweep, then peer review

**Task:** run t-SNE at three perplexities and see how the picture changes.
Recall that perplexity controls the effective number of neighbors (typical range
5-50, default 30) and must be smaller than the number of points.

In [ ]:
# Uses the 50-dimensional PCA representation for speed
perplexities = [5, 30, 50]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, perp in zip(axes, perplexities):
    emb = TSNE(n_components=2, perplexity=perp, random_state=42,
               init='pca').fit_transform(X_dig_pca50)
    ax.scatter(emb[:, 0], emb[:, 1], c=y_dig, cmap='tab10', s=10, alpha=0.8)
    ax.set_title(f't-SNE (perplexity={perp})')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

plt.tight_layout()
plt.show()

<!-- BEGIN QUESTION -->

**Discussion question:** which structures survive across all three perplexities,
and which appear or vanish? Those that survive are the ones you are entitled to
talk about.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

## Peer review

1. **Find another group** to pair up with.
2. **Compare:** did you pick the same `n_components_90`? The same reading of the
   digits plots?
3. **Argue:** if you disagree on any answer in Part 6, work out which of you can
   point at evidence in the plots.
4. **Report back:** be ready to state your group's Part 6 answer to the class.